# HealthPredictor
<hr>

## Final Jupyter Notebook, Group 1 - Healthcare
## New Dataset
## CIS-579-002, Introduction to Artificial Intelligence
### Avinash Shete, Chandana Bhadravati Nagaraj, Jim Small, Ritesh Revansiddappa Honnalli

### Setup environment/notebook:

In [ ]:
# Needed libraries:
# Standard Library:
from functools import partial
import math
import pickle
import shelve
from textwrap import fill

# 3rd Party:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display as display
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# Suppress all warnings - remove for debugging/tuning:
# Note:  Only warning is from cell 34 because resulting plot is only of 1 class!
# import warnings
# warnings.filterwarnings('ignore')

plt.style.use('fivethirtyeight')
%matplotlib inline
pd.set_option('display.max_columns', 54)

# Track run number for serial execution:
# run_id = # is passed in
run_file = 'hp_results'

# Imputation type for missing numerical values:
# Either random or knn
IMPUTATION = 'knn'

In [ ]:
# Load data:
# Note:  Using forward slashes ("/" versus "\") so works on both Windows and Linux/macOS:
data = './Data/chronickidneydisease-kaggle-1659x52.csv'
df = pd.read_csv(data)

# Explore first few rows:
df.head()

### First look at data/dataframe:

In [ ]:
# Dimensions (rows, columns) of dataframe:
df.shape

In [ ]:
# Show overview of dataframe columns:
df.info()

In [ ]:
# Show number of unique values per column:
df.nunique()

In [ ]:
# Examine descriptive statistics for each column - average, standard deviation, quartiles, and more:
df.describe()

### Data Preprocessing:

In [ ]:
# Save original DataFrame to allow extracting records by PatientID later on:
orig_df = df.copy()

# Remove id column:
df.drop('PatientID', axis=1, inplace=True)

# Remove doctor column - "Confidential" for all patients:
df.drop('DoctorInCharge', axis=1, inplace=True)

In [ ]:
# Again look at first few rows to see column name changes:
df.head()

In [ ]:
# Categorical Columns:
# * Gender: Male=0, Female=1
# * Ethnicity: 0=Caucasian, 1=African American, 2=Asian, 3=Other
# * SocioeconomicStatus: 0=Low, 1=Middle, 2=High
# * EducationLevel: 0=None, 1=High School, 2=Bachelor's, 3=Higher
# * Smoking: 0=No, 1=Yes
# * FamilyHistoryKidneyDisease: 0=No, 1=Yes
# * FamilyHistoryHypertension: 0=No, 1=Yes
# * FamilyHistoryDiabetes: 0=No, 1=Yes
# * PreviousAcuteKidneyInjury: 0=No, 1=Yes
# * UrinaryTractInfections: 0=No, 1=Yes
# * ACEInhibitors: 0=No, 1=Yes
# * Diuretics: 0=No, 1=Yes
# * Statins: 0=No, 1=Yes
# * AntidiabeticMedications: 0=No, 1=Yes
# * Edema: 0=No, 1=Yes
# * HeavyMetalsExposure: 0=No, 1=Yes
# * OccupationalExposureChemicals: 0=No, 1=Yes
# * WaterQuality: 0=Good, 1=Poor
# * Diagnosis: 0=No, 1=Yes for CKD

# Create categorical dataframe:
cat_df = pd.DataFrame()

# Define the mappings from int64 -> category:
mappings = {
    'Gender': {0: 'Male', 1: 'Female'},
    'Ethnicity': {0: 'Caucasian', 1: 'African American', 2: 'Asian', 3: 'Other'},
    'SocioeconomicStatus': {0: 'Low', 1: 'Middle', 2: 'High'},
    'EducationLevel': {0: 'None', 1: 'High School', 2: "Bachelor's", 3: 'Higher'},
    'Smoking': {0: 'No', 1: 'Yes'},
    'FamilyHistoryKidneyDisease': {0: 'No', 1: 'Yes'},
    'FamilyHistoryHypertension': {0: 'No', 1: 'Yes'},
    'FamilyHistoryDiabetes': {0: 'No', 1: 'Yes'},
    'PreviousAcuteKidneyInjury': {0: 'No', 1: 'Yes'},
    'UrinaryTractInfections': {0: 'No', 1: 'Yes'},
    'ACEInhibitors': {0: 'No', 1: 'Yes'},
    'Diuretics': {0: 'No', 1: 'Yes'},
    'Statins': {0: 'No', 1: 'Yes'},
    'AntidiabeticMedications': {0: 'No', 1: 'Yes'},
    'Edema': {0: 'No', 1: 'Yes'},
    'HeavyMetalsExposure': {0: 'No', 1: 'Yes'},
    'OccupationalExposureChemicals': {0: 'No', 1: 'Yes'},
    'WaterQuality': {0: 'Good', 1: 'Poor'},
    'Diagnosis': {0: 'No-CKD', 1: 'CKD'},
}

# Apply the mapping and convert to categorical
for col, mapping in mappings.items():
    cat_df[col] = df[col].map(mapping).astype('category')

# Check results
print(f'Categorical Data/Columns:\n{'=' * 90}\n{cat_df.dtypes}\n')
display(cat_df.head())

# Numerical Columns:
# * Age: 20 - 90
# * BMI: 15 - 40
# * AlcoholConsumption: 0 - 20 alchohol units/week
# * PhysicalActivity: 0 - 10 hours/week
# * DietQuality: 0 - 10 score
# * SleepQuality: 4 - 10 score
# * SystolicBP: 90 - 180
# * DiastolicBP: 60 - 120
# * FastingBloodSugar: 70 - 200
# * HbA1c: 4.0% - 10.0%
# * SerumCreatinine: 0.5 - 5.0
# * BUNLevels: 5 - 50
# * GFR: 15 - 120
# * ProteinInUrine: 0 - 5
# * ACR: 0 - 300
# * SerumElectrolytesSodium: 135 - 145
# * SerumElectrolytesPotassium: 3.5 - 5.5
# * SerumElectrolytesCalcium: 8.5 - 10.5
# * SerumElectrolytesPhosphorus: 2.5 - 4.5
# * HemoglobinLevels: 10 - 18
# * CholesterolTotal: 150 - 300
# * CholesterolLDL: 50 - 200
# * CholesterolHDL: 20 - 100
# * CholesterolTriglycerides: 50 - 400
# * NSAIDsUse: 0 - 10 times/week
# * FatigueLevels: 0 - 10
# * NauseaVomiting: 0 - 7 times/week
# * MuscleCramps: 0 - 7 times/week
# * Itching: 0 - 10 severity
# * QualityOfLifeScore: 0 - 100
# * MedicalCheckupsFrequency: 0 - 4 per year
# * MedicationAdherence: 0 - 10
# * HealthLiteracy: 0 - 10

# Create numerical dataframe:
num_columns = [
    'Age', 'BMI', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality', 'SleepQuality',
    'SystolicBP', 'DiastolicBP', 'FastingBloodSugar', 'HbA1c', 'SerumCreatinine', 'BUNLevels',
    'GFR', 'ProteinInUrine', 'ACR', 'SerumElectrolytesSodium', 'SerumElectrolytesPotassium',
    'SerumElectrolytesCalcium', 'SerumElectrolytesPhosphorus', 'HemoglobinLevels',
    'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides',
    'NSAIDsUse', 'FatigueLevels', 'NauseaVomiting', 'MuscleCramps', 'Itching',
    'QualityOfLifeScore', 'MedicalCheckupsFrequency', 'MedicationAdherence', 'HealthLiteracy'
]
num_df = df[num_columns]

# Check results
print(f'\n\nNumerical Data/Columns:\n{'=' * 90}\n{num_df.dtypes}\n')
display(num_df.head())

### Visually Explore Data:

In [ ]:
len(df.columns)

In [ ]:
# Visualize numerical features distribution:

# Determine number of plots and subplot grid:
num_plots = len(num_df.columns)
cols = 5
rows = math.ceil(num_plots / cols)

cell_width = 5  # inches per subplot column
cell_height = 4  # inches per subplot row

# Scale figure size with number of rows
plt.figure(figsize=(cols * cell_width, rows * cell_height))

for i, column in enumerate(sorted(num_df.columns), 1):
    ax = plt.subplot(rows, cols, i)
    # Try to use histplot instead of deprecated distplot:
    # sns.histplot(df[column], kde=True, ax=ax)
    # sns.distplot(df[column])
    sns.histplot(num_df[column], kde=True, stat="density", element="bars", ax=ax)
    # Optional styling:
    # sns.rugplot(df[column], ax=ax, color="black", height=0.05)
    ax.set_xlabel(column)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize categorical column data distribution:

num_plots = len(cat_df.columns)
cols = 3
rows = math.ceil(num_plots / cols)

cell_width = 5  # inches per subplot column
cell_height = 4  # inches per subplot row

plt.figure(figsize=(cols * cell_width, rows * cell_height))

for i, column in enumerate(sorted(cat_df.columns), 1):
    ax = plt.subplot(rows, cols, i)
    sns.countplot(data=cat_df, x=column, hue=column, palette='rocket', legend=False, ax=ax)
    ax.set_xlabel(column)
    # 🔄 Rotate x-axis labels to fit in view:
    for label in ax.get_xticklabels():
        label.set_rotation(45)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of data:

# Take numerica data and CKD diagnosis:
hm_df = num_df.copy()

# Sort columns and add diagnosis:
hm_cols = sorted(hm_df.columns)
hm_df = hm_df[hm_cols]
hm_df['Diagnosis'] = df['Diagnosis']

cell_size = 1.25  # in inches — adjust to make text readable
columns = hm_df.shape[1]
rows = columns  # square correlation matrix

width = cell_size * columns
height = cell_size * rows

plt.figure(figsize=(width, height))
sns.heatmap(hm_df.corr(numeric_only=True), annot=True, linewidths=2, linecolor='lightgrey')

plt.show()

### Feature Selection:

In [ ]:
# Helper display function for long lists:
def wrap(output: list[str], width: int=80) -> str:
    return fill(', '.join(output), width=width)

In [ ]:
cutoff = 0.05

# Take numerica data and CKD diagnosis:
cor_df = num_df.copy()

# Sort columns and add diagnosis:
cor_cols = sorted(cor_df.columns)
cor_df = cor_df[cor_cols]
cor_df['Diagnosis'] = df['Diagnosis']

# For numeric columns, only keep those with an absolute correlation > cutoff:
correlations = cor_df.corrwith(cor_df['Diagnosis'])
cor_num_cols = correlations[correlations.abs() > cutoff].index.tolist()
# Remove 'Diagnosis' - not numeric column:
cor_num_cols.pop()

# Original columns:
orig_num_cols = sorted(num_df.columns)
print(f'Original numeric features:\n{wrap(orig_num_cols)}')

# Update dataframe:
num_df = num_df[cor_num_cols]
upd_num_cols = sorted(num_df.columns)

# Removed columns:
rem_num_cols = sorted(set(orig_num_cols) - set(upd_num_cols))
print(f'\nRemoved numeric features:\n{wrap(rem_num_cols)}')

# Resulting columns:
print(f'\nResulting numeric features:\n{wrap(sorted(num_df.columns))}')

In [ ]:
from sklearn.feature_selection import f_classif, mutual_info_classif

# Load, prep, split features and outcome
# cat_df already exists
# num_df already exists
y = df['Diagnosis']

# Encode categorical outcome (if needed)
if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y)

# Select only numeric features
X_numeric = num_df.select_dtypes(include='number')

# ANOVA F-test
f_scores, p_values = f_classif(X_numeric, y)
f_test_results = pd.DataFrame({
    'feature': X_numeric.columns,
    'f_score': f_scores,
    'p_value': p_values
}).sort_values(by='f_score', ascending=False)

print(f'ANOVA F-test results:')
display(f_test_results)

# Looking for:
# * High F-score → The feature is likely useful for classification
# * Low p-value (e.g., < 0.05) → The feature’s relationship with the outcome is statistically significant
#
# Filter the rows where p_value < 0.05:
rel_num_cols1 = f_test_results[f_test_results['p_value'] < 0.05]['feature'].tolist()
print(f'Significant/relevant features selected:\n* {"\n* ".join(rel_num_cols1)}')

# Mutual Information
mi_scores = mutual_info_classif(X_numeric, y)
mi_results = pd.Series(mi_scores, index=X_numeric.columns).sort_values(ascending=False)

print('\nMutual Information scores:')
display(mi_results)

# Looking for:
# * Mutual Information used to rank features in terms of value (highest to lowest)
#
# Filter the rows where the socre is 0 (< 0.0000001):
rel_num_cols2 = mi_results[mi_results > 1e-7].index.tolist()
print(f'\n\nSignificant/relevant features selected:\n* {"\n* ".join(rel_num_cols2)}')

# Final column selection:
rel_num_cols = list(set(rel_num_cols1) | set(rel_num_cols2))

# Original columns:
orig_num_cols = sorted(rel_num_cols)
print(f'\nOriginal numeric features:\n{wrap(orig_num_cols)}')

# Update dataframe:
num_df = num_df[rel_num_cols]
upd_num_cols = sorted(num_df.columns)

# Removed columns:
rem_num_cols = sorted(set(orig_num_cols) - set(upd_num_cols))
if rem_num_cols:
    print(f'\nRemoved numeric features:\n{wrap(rem_num_cols)}')
else:
    print('\nNo numeric features removed.')

# Resulting columns:
print(f'\nResulting numeric features:\n{wrap(sorted(num_df.columns))}')

In [ ]:
from sklearn.feature_selection import chi2
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer

# Load, prep, split features and outcome
# cat_df already exists
# num_df already exists
# y = df['Diagnosis'] already exists
X = cat_df.drop(columns=['Diagnosis'])

# Encode outcome if needed
if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y)

# Select categorical features
# X_cat = X.select_dtypes(include='object').copy()

# One-hot encode categorical features
X_encoded = pd.get_dummies(X)

# Run chi-squared test
chi_scores, p_values = chi2(X_encoded, y)
chi2_results = pd.DataFrame({
    'feature': X_encoded.columns,
    'chi2_score': chi_scores,
    'p_value': p_values
}).sort_values(by='chi2_score', ascending=False)

print('Chi-squared test results:')
display(chi2_results)

# Final column selection:
rel_cat_cols = chi2_results[
    (chi2_results['chi2_score'] >= 3) & (chi2_results['p_value'] < 0.10)
]['feature'].tolist()

# Original columns:
orig_num_cols = sorted(cat_df.columns)
print(f'\nOriginal categorical features:\n{wrap(orig_num_cols)}')

# Update dataframe:
rel_cat_cols = list({col.split('_')[0] for col in rel_cat_cols})
cat_df = cat_df[rel_cat_cols]
upd_num_cols = sorted(cat_df.columns)

# Removed columns:
rem_num_cols = sorted(set(orig_num_cols) - set(upd_num_cols))
print(f'\nRemoved categorical features:\n{wrap(rem_num_cols)}')

# Resulting columns:
print(f'\nResulting categorical features:\n{wrap(sorted(cat_df.columns))}')

In [ ]:
# Original columns:
orig_cols = sorted(df.columns)
print(f'Original dataset features:\n{wrap(orig_cols)}')

# Diagnosis column:
ckd_col_list = ['Diagnosis']

# Create updated dataframe columns:
rel_cols = rel_num_cols + rel_cat_cols + ckd_col_list

# Update dataframe:
df = df[rel_cols]
upd_cols = sorted(df.columns)

# Removed columns:
rem_cols = sorted(set(orig_cols) - set(upd_cols))
print(f'\nRemoved dataset features:\n{wrap(rem_cols)}')

# Resulting columns:
print(f'\nResulting dataset features:\n{wrap(sorted(df.columns))}')

In [ ]:
# Revisit Heatmap of data:

# Take numerica data and CKD diagnosis:
hm_df = num_df.copy()
hm_df['Diagnosis'] = df['Diagnosis']

cell_size = 1.25  # in inches — adjust to make text readable
columns = hm_df.shape[1]
rows = columns  # square correlation matrix

width = cell_size * columns
height = cell_size * rows

plt.figure(figsize=(width, height))
sns.heatmap(hm_df.corr(numeric_only=True), annot=True, linewidths=2, linecolor='lightgrey')

plt.show()

### Data Cleaning:

In [ ]:
# Check for null values:
df.isna().sum().sort_values(ascending = False)

### Feature Encoding:

In [ ]:
# Not needed - all values began as numerical

### Label Encoding:

In [ ]:
# Not needed - all values began as numerical

<a id = '5.0'></a>
<p style = "font-size : 45px; color : #34656d ; font-family : 'Comic Sans MS'; text-align : center; background-color : #f9b208; border-radius: 5px 5px;"><strong>Model Building</strong></p> 

In [ ]:
ckd_col = 'Diagnosis'

ind_col = [col for col in df.columns if col != ckd_col]
dep_col = ckd_col

X = df[ind_col]
y = df[dep_col]

In [ ]:
# Split data into training and test sets:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30, random_state = 0)

In [ ]:
# Build confusion matrix results dataframe:
def get_cm_results(cm):
    # cm is the result of sklearn.metrics.confusion_matrix(y_test, classifier.predict(X_test))
    tn, fp, fn, tp = cm.ravel()
    data = [
        {'Result': 'True Positive', 'Number': tp,
         'Description': 'Actual occurrence, correctly predicted'},
        {'Result': 'True Negative', 'Number': tn,
         'Description': 'Non-occurrence, correctly predicted'},
        {'Result': 'False Negative', 'Number': fn,
         'Description': 'Actual occurrence, incorrectly predicted (Type II error)'},
        {'Result': 'False Positive', 'Number': fp,
         'Description': 'Non-occurrence, incorrectly predicted (Type I error)'},
    ]
    return pd.DataFrame(data).style.set_properties(
        subset=['Description'], **{'text-align': 'left'}
    ).hide(axis='index')


# Build confusion matrix metrics dataframe:
def get_cm_metrics(cm):
    # cm is the result of sklearn.metrics.confusion_matrix(y_test, classifier.predict(X_test))
    tn, fp, fn, tp = cm.ravel()
    prec_val = tp/(tp + fp)
    rec_val = tp/(tp + fn)
    data = [
        {'Metric': 'Accuracy', 'Value': (tp + tn)/(tp + tn + fp + fn),
         'Description': 'How often is classifier correct'},
        {'Metric': 'Precision', 'Value': prec_val,
         'Description': 'How often is TP correctly predicted (% of correct positives)'},
        {'Metric': 'Recall', 'Value': rec_val,
         'Description': 'How often is actual occurrence correctly predicted'},
        {'Metric': 'F1-Score', 'Value': (2 * prec_val * rec_val)/(prec_val + rec_val),
         'Description': 'Account for both precision and recall using harmonic mean'},
        {'Metric': 'Misclassification', 'Value': (fp + fn)/(tp + tn + fp + fn),
         'Description': 'How often is classifier wrong'},
        {'Metric': 'NPV', 'Value': tn/(tn + fn),
         'Description': 'How often is TN correctly predicted (% of correct negatives)'},
        {'Metric': 'FPR', 'Value': fp/(tn + fp),
         'Description': 'How often is non-occurrence falsely predicted'},
        {'Metric': 'Specificity', 'Value': tn/(tn + fp),
         'Description': 'How often is actual non-occurrence correctly predicted'},
        {'Metric': 'Prevalence', 'Value': (fn + tp)/(tp + tn + fp + fn),
         'Description': 'How often does actually occur in sample'},
    ]
    return pd.DataFrame(data).style.set_properties(
        subset=['Description'], **{'text-align': 'left'}
    ).format({'Value': '{:.2%}'}).hide(axis='index')

# Plot confusion matrix heatmap:
def cm_heatmap(cm):
    classes = ['No CKD', 'CKD']
    df_cm = pd.DataFrame(cm, index=classes, columns=classes)
    
    # Flip horizontally and vertically
    df_cm = df_cm.iloc[::-1, ::-1]
    
    plt.figure(figsize=(5,4))
    sns.heatmap(df_cm, annot=True, fmt='d', cmap='coolwarm_r',
                linewidths=0.5, cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix Heatmap')
    return plt.show()

# Track results by saving/serializing:
def save_cm(cm, model):
    # cm is the result of sklearn.metrics.confusion_matrix(y_test, classifier.predict(X_test))
    tn, fp, fn, tp = cm.ravel()
    prec_val = tp/(tp + fp)
    rec_val = tp/(tp + fn)
    accuracy = (tp + tn)/(tp + tn + fp + fn)
    precision = prec_val
    recall = rec_val
    f1_score = (2 * prec_val * rec_val)/(prec_val + rec_val)
    misclassification = (fp + fn)/(tp + tn + fp + fn)
    npv = tn/(tn + fn)
    fpr = fp/(tn + fp)
    specificity = tn/(tn + fp)
    prevalence = (fn + tp)/(tp + tn + fp + fn)
    model_name = model if isinstance(model, str) else model.__class__.__name__
    imputation = IMPUTATION

    # Safety check in case Notebook run directly:
    run_id = globals().get('run_id', 0)
    with shelve.open(run_file) as db:
        db[f'{run_id}/{model_name}'] = dict(
            true_positive=tp, true_negative=tn, false_negative=fn, false_postive=fp,
            accuracy=accuracy, precision=precision, recall=recall, f1_score=f1_score,
            misclassification=misclassification, npv=npv, fpr=fpr, specificity=specificity,
            prevalence=prevalence, imputation=imputation
        )

<a id = '5.2'></a>
<p style = "font-size : 25px; color : #34656d ; font-family : 'Comic Sans MS'; text-align : center; background-color : #fbc6a4; border-radius: 5px 5px;"><strong>Decision Tree Classifier</strong></p> 

In [ ]:
dtc = DecisionTreeClassifier()
dtc.fit(X_train, y_train)

# Accuracy score, confusion matrix and classification report of decision tree:
dtc_acc = accuracy_score(y_test, dtc.predict(X_test))

print(f"Training Accuracy of Decision Tree Classifier is {accuracy_score(y_train, dtc.predict(X_train))}")
print(f"Test Accuracy of Decision Tree Classifier is {dtc_acc} \n")

cm = confusion_matrix(y_test, dtc.predict(X_test))
print(f"Confusion Matrix :- \n{cm}\n")
print(f"Classification Report :- \n {classification_report(y_test, dtc.predict(X_test))}")

In [ ]:
# Plot confusion matrix heatmap:
cm_heatmap(cm)

# Show classifier results and metrics:
display(get_cm_results(cm))
display(get_cm_metrics(cm))

# Track results:
save_cm(cm, dtc)

In [ ]:
# Hyper parameter tuning of decision tree:

# Input hyper paramter to choose from to find the best estimator to increase the accuracy
# and avoid overfitting of the model:
grid_param = {
    'criterion' : ['gini', 'entropy'],
    'max_depth' : [3, 5, 7, 10],
    'splitter' : ['best', 'random'],
    'min_samples_leaf' : [1, 2, 3, 5, 7],
    'min_samples_split' : [2, 3, 5, 7],
    'max_features' : ['sqrt', 'log2']  # How many input features the tree considers to build the model
}

# Use GridSearchCV to try provided different compbination of hyper parameter:
grid_search_dtc = GridSearchCV(dtc, grid_param, cv = 5, n_jobs = -1, verbose = 1)
grid_search_dtc.fit(X_train, y_train)

In [ ]:
# Best parameters and best score:
print(grid_search_dtc.best_params_)
print(grid_search_dtc.best_score_)

In [ ]:
# Best estimator:
dtc = grid_search_dtc.best_estimator_

# Accuracy score, confusion matrix and classification report of decision tree
dtc_acc = accuracy_score(y_test, dtc.predict(X_test))

print(
    "Training Accuracy of Decision Tree Classifier - Hyperparameter best estimator is "
      f"{accuracy_score(y_train, dtc.predict(X_train))}"
)
print(f"Test Accuracy of Decision Tree Classifier - Hyperparameter best estimator is {dtc_acc} \n")

cm = confusion_matrix(y_test, dtc.predict(X_test))
print(f"Confusion Matrix after hyperparameter tuning :- \n{cm}\n")
print(
    "Classification Report after hyperparameter tuning :- \n "
    f"{classification_report(y_test, dtc.predict(X_test))}"
)

In [ ]:
# Plot confusion matrix heatmap:
cm_heatmap(cm)

# Show classifier results and metrics:
display(get_cm_results(cm))
display(get_cm_metrics(cm))

# Track results:
save_cm(cm, 'DecisionTreeClassifier-GridSearchCV')

<a id = '5.3'></a>
<p style = "font-size : 25px; color : #34656d ; font-family : 'Comic Sans MS'; text-align : center; background-color : #fbc6a4; border-radius: 5px 5px;"><strong>Random Forest Classifier</strong></p>

In [ ]:
# Note:  The default for max_features changed from 'auto' to 'sqrt', updating:
# Direct initialization of RandomForestClassifier object for specific hyperparamter value.
# Not using RandomizedSearchCV for the tunning.
rd_clf = RandomForestClassifier(
    criterion = 'entropy', max_depth = 11, max_features = 'sqrt',
    min_samples_leaf = 2, min_samples_split = 3, n_estimators = 130
)
rd_clf.fit(X_train, y_train)

# Accuracy score, confusion matrix and classification report of random forest:
rd_clf_acc = accuracy_score(y_test, rd_clf.predict(X_test))

print(f"Training Accuracy of Random Forest Classifier is {accuracy_score(y_train, rd_clf.predict(X_train))}")
print(f"Test Accuracy of Random Forest Classifier is {rd_clf_acc} \n")

cm = confusion_matrix(y_test, rd_clf.predict(X_test))
print(f"Confusion Matrix :- \n{cm}\n")
print(
    "Classification Report :- \n "
    # Can change zero_division parameter to 0 to suppress warning:
    # This happens when TP & FP = 0 and/or TP & FN = 0
    f"{classification_report(y_test, rd_clf.predict(X_test), zero_division='warn')}"
)

In [ ]:
# Plot confusion matrix heatmap:
cm_heatmap(cm)

# Show classifier results and metrics:
display(get_cm_results(cm))
display(get_cm_metrics(cm))

# Track results:
save_cm(cm, rd_clf)

<a id = '5.3'></a>
<p style = "font-size : 25px; color : #34656d ; font-family : 'Comic Sans MS'; text-align : center; background-color : #fbc6a4; border-radius: 5px 5px;"><strong>XgBoost</strong></p>

In [ ]:
# Use of the XGBoost (Extreme Gradient Boosting) algorithm for a binary classification task
xgb = XGBClassifier(objective = 'binary:logistic', learning_rate = 0.5, max_depth = 5, n_estimators = 150)
xgb.fit(X_train, y_train)

# accuracy score, confusion matrix and classification report of xgboost
xgb_acc = accuracy_score(y_test, xgb.predict(X_test))

print(f"Training Accuracy of XgBoost is {accuracy_score(y_train, xgb.predict(X_train))}")
print(f"Test Accuracy of XgBoost is {xgb_acc} \n")

cm = confusion_matrix(y_test, xgb.predict(X_test))
print(f"Confusion Matrix :- \n{cm}\n")
print(f"Classification Report :- \n {classification_report(y_test, xgb.predict(X_test))}")

In [ ]:
# Plot confusion matrix heatmap:
cm_heatmap(cm)

# Show classifier results and metrics:
display(get_cm_results(cm))
display(get_cm_metrics(cm))

# Track results:
save_cm(cm, xgb)

<a id = '5.3'></a>
<p style = "font-size : 25px; color : #34656d ; font-family : 'Comic Sans MS'; text-align : center; background-color : #fbc6a4; border-radius: 5px 5px;"><strong>Logistic Regression</strong></p>

In [ ]:
# Use of the Logistic Regression algorithm for a binary classification task
# liblinear is good for smaller datasets. For larger datasets, solvers like 'lbfgs'
# or 'saga' may perform better.  This parameter sets the seed for the random number
# generator used by the model. Setting a random_state ensures that the results are
# reproducible. If you run the code multiple times with the same random_state, you'll
# get the same results.
logistic_regression = LogisticRegression(solver='liblinear', random_state=42) 

logistic_regression.fit(X_train, y_train)

# accuracy score, confusion matrix and classification report of logistic regression
logistic_acc = accuracy_score(y_test, logistic_regression.predict(X_test))

print(
    "Training Accuracy of Logistic Regression is "
    f"{accuracy_score(y_train, logistic_regression.predict(X_train))}"
)
print(f"Test Accuracy of Logistic Regression is {logistic_acc} \n")

cm = confusion_matrix(y_test, logistic_regression.predict(X_test))
print(f"Confusion Matrix :- \n{cm}\n")
print(
    "Classification Report :- \n "
    f"{classification_report(y_test, logistic_regression.predict(X_test))}"
)

In [ ]:
# Plot confusion matrix heatmap:
cm_heatmap(cm)

# Show classifier results and metrics:
display(get_cm_results(cm))
display(get_cm_metrics(cm))

# Track results:
save_cm(cm, logistic_regression)

<a id = '6.0'></a>
<p style = "font-size : 35px; color : #34656d ; font-family : 'Comic Sans MS'; text-align : center; background-color : #f9b208; border-radius: 5px 5px;"><strong>Models Comparison</strong></p> 

In [ ]:
models = pd.DataFrame({
    'Model' : ['Decision Tree Classifier', 'Random Forest Classifier', 'XgBoost', 'Logistic Regression'],
    'Score' : [dtc_acc, rd_clf_acc, xgb_acc, logistic_acc]
})

models.sort_values(by='Score', ascending=False)

In [ ]:
# Flip sort order to show best model at the top:
models_sorted = models.sort_values(by='Score', ascending=True)

fig = px.bar(
    data_frame=models_sorted,
    x='Score',
    y='Model',
    color='Score',
    template='plotly_dark',
    title='Models Comparison'
)
fig.show()

### Select XGBoost model - more reliable:

In [ ]:
# Top 10 Features:
feature_scores = pd.DataFrame(
    xgb.feature_importances_, columns=['Score'], index=X_train.columns
).sort_values(by='Score', ascending=False)
top10_features = feature_scores.nlargest(n=10, columns=['Score'])

plt.figure(figsize=(14, 8))
g = sns.barplot(x=top10_features.index, y=top10_features['Score'])
p = plt.title('Top 10 Features with XGBoost')
p = plt.xlabel('Feature name')
p = plt.ylabel('XGBoost score')

# Rotate x-axis tick labels
for tick in g.get_xticklabels():
    tick.set_rotation(45)
    tick.set_horizontalalignment('right')

plt.tight_layout()
plt.show()

In [ ]:
top10_features = top10_features.index.tolist()
top10_features

In [ ]:
# Current columns:
X.columns

In [ ]:
# Prune columns not in top 10:
for ele in X.columns:
    if ele not in top10_features:
        X = X.drop(ele, axis = 1)

# Display:
X.head()

In [ ]:
X_train=X_train[top10_features]
X_test=X_test[top10_features]
xgb.fit(X_train, y_train)

### Testing Predictions:

In [ ]:
# XGBoost feature ordering must match - training order and patient data order
def get_ordered_pd_df(df: pd.DataFrame, patient_id: int, columns: list[str]) -> pd.DataFrame:
    """
    Extract a single row from the DataFrame by PatientID and return only the selected
    columns in the specified order.
    
    Parameters:
    - df: The source DataFrame.
    - patient_id: The PatientID identifying the row.
    - columns: A list of columns to include in the returned row (in order).
    
    Returns:
    - A DataFrame with one row and only the specified columns.
    """
    return df.loc[df['PatientID'] == patient_id, columns]

In [ ]:
# Prediction 1 - CKD:
patient_id = 7

# Use Original DataFrame which retains PatientID column:
pd_df = get_ordered_pd_df(orig_df, patient_id, top10_features)
prediction = xgb.predict(pd_df)[0]

if prediction:
    print('Unfortunately, you have Chronic Kidney Disease.')
else:
    print("Fortunately, you don't have Chronic Kidney Disease.")

In [ ]:
# Prediction 2 - No CKD:
patient_id = 8

# Use Original DataFrame which retains PatientID column:
pd_df = get_ordered_pd_df(orig_df, patient_id, top10_features)
prediction = xgb.predict(pd_df)[0]

if prediction:
    print('Unfortunately, you have Chronic Kidney Disease.')
else:
    print("Fortunately, you don't have Chronic Kidney Disease.")

### Serialize and save model:

In [ ]:
pickle.dump(xgb, open('CKD-XGB-Kaggle.pkl', 'wb'))